# Model 02 — Naive Bayes (Surface Defect Inspection — Painted Plastic Parts)

Surface defects in painted plastic parts are one of the most expensive quality problems in manufacturing — not because they are structurally dangerous, but because they are highly visible and impossible to hide. A painted automotive trim piece with a scratch, a blister, or a dust inclusion goes directly to scrap or rework. The cost is not just material; it is labor, cycle time, and the downstream disruption of holding parts at the line.

The process behind this dataset has four stages: **injection molding → internal storage and transport → paint cabin → final inspection**. Each stage introduces its own defect risk, but they do not act in isolation. A piece that exits injection in acceptable condition can be degraded by rough handling, excessive waiting time, or inadequate protection — and those defects become visible only after paint is applied. This delayed revelation is what makes surface defect prediction challenging: by the time the defect is detected, the damage was done hours or stages earlier.

**Naive Bayes** is the right model for this type of problem. It assumes feature independence — which is a reasonable approximation when mixing process variables from different stages — and it returns calibrated probabilities that translate directly into inspection risk scores. More importantly, it forces the analyst to look at each variable's *individual* contribution to the outcome, which aligns well with how quality engineers approach root cause analysis: one factor at a time, quantified by effect size.

---
**Data source:** `surface_defect_inspection_data.csv`  
**Output:** metrics, Cohen's d effect sizes, Digital Pareto, and a part inspection simulator


## Section 1 — Setup

Reproducibility ensures that every run of this notebook produces the same results — essential when sharing findings with quality or process engineering teams. We fix a global seed, import all dependencies upfront, and configure visualization defaults once.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
from sklearn.feature_selection import mutual_info_classif
from scipy import stats

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Libraries loaded. Random state:", RANDOM_STATE)


## Section 2 — Load Data

The dataset contains **1,500 simulated inspection records** for painted plastic parts. Each row represents one part, tracked through the full production sequence. The 10 input features span three process stages: injection (resin temperature, regrind percentage, cooling time), paint cabin (viscosity, film thickness, booth humidity), and internal logistics (pre-paint storage time, number of handlings, container type, part protection). The target `surface_defect` is binary — 1 if the part showed a visual defect at final inspection, 0 if it passed.

> *Note: The raw dataset in this project was generated using a simulation script that models realistic plastic part manufacturing conditions — where transport, handling, and pre-paint waiting time are the dominant defect drivers, interacting with paint cabin conditions to produce a defect rate of approximately 21.9%. The `.csv` file used here is the final output of that process, loaded directly as you would from a real MES or quality system export.*


In [ ]:
try:
    df = pd.read_csv("surface_defect_inspection_data.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/02-Naive-Bayes-Surface-Defect/main/surface_defect_inspection_data.csv")
    # FileNotFoundError is intentionally specific — a bare except would silently swallow real data errors

df.head()


## Section 3 — Quick Sanity Checks

Before fitting any model, we validate the data we actually loaded. The checks below confirm shape, missing values, data types, and class balance. With a ~22% defect rate the dataset is moderately imbalanced — not severe enough to require resampling, but enough to make accuracy a somewhat optimistic metric. Recall for the defect class is the number that matters most operationally.


In [ ]:
# Sanity checks. Real datasets usually try to hurt you 🙂
print("Shape:", df.shape)
print("\n--- Data types ---")
df.info()


In [ ]:
print("--- Missing values ---")
print(df.isna().sum())
print("\n--- Target class balance ---")
print(df["surface_defect"].value_counts())
print("\n--- Defect rate ---")
print(df["surface_defect"].value_counts(normalize=True).rename({0: "pass", 1: "defect"}))


In [ ]:
print("--- Categorical variables ---")
print("container_type   :", df["container_type"].value_counts().to_dict())
print("part_protection  :", df["part_protection"].value_counts().to_dict())
print("\n--- Numeric summary ---")
df.describe().round(2)


## Section 4 — Exploratory Data Analysis

EDA here is about hypothesis formation, not decoration. Before training, we want to understand which process conditions are visually associated with defects — so we can validate that the model learns real structure, not noise. We organize the analysis around three questions:

1. **How do numeric features distribute differently between defect and non-defect parts?** — boxplots by class reveal median shift and variance spread.
2. **Which categorical conditions concentrate the most defects?** — bar charts of defect rate by container type and protection status.
3. **Are there correlation patterns between features and the target?** — a focused heatmap shows which variables have the strongest linear association with `surface_defect`.

The patterns found here should map directly to what the Naive Bayes model captures in Section 8.


In [ ]:
# Define feature groups — used throughout the notebook
NUM_COLS = [
    "regrind_pct", "resin_temp_c", "cooling_time_s",
    "paint_viscosity", "film_thickness_um", "booth_humidity_pct",
    "pre_paint_storage_hrs", "num_handlings"
]
CAT_COLS = ["container_type", "part_protection"]
TARGET   = "surface_defect"


In [ ]:
# Numeric distributions — looking for skew and outliers before scaling
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(NUM_COLS):
    axes[i].hist(df[df[TARGET]==0][col], bins=25, alpha=0.6,
                 color="#4C9BE8", edgecolor="white", label="Pass")
    axes[i].hist(df[df[TARGET]==1][col], bins=25, alpha=0.6,
                 color="#E8574C", edgecolor="white", label="Defect")
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].legend(fontsize=8)
plt.suptitle("Numeric Feature Distributions by Defect Status",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots by class — where do the medians and spreads diverge?
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(NUM_COLS):
    for val, color in [(0, "#4C9BE8"), (1, "#E8574C")]:
        subset = df[df[TARGET] == val][col]
        axes[i].boxplot([subset.values], positions=[val],
                        patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.7),
                        medianprops=dict(color="white", linewidth=2))
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(["Pass", "Defect"])
plt.suptitle("Feature Distribution by Defect Class (Blue = Pass, Red = Defect)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Categorical breakdown — defect rate by container type and protection
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, CAT_COLS):
    rates = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    bars = ax.bar(rates.index, rates.values,
                  color=["#E8574C" if v > df[TARGET].mean() else "#4C9BE8"
                         for v in rates.values],
                  edgecolor="white")
    ax.axhline(df[TARGET].mean(), linestyle="--", color="gray",
               linewidth=1.2, label=f"Overall avg {df[TARGET].mean():.1%}")
    ax.set_title(f"Defect Rate by {col.replace('_', ' ').title()}")
    ax.set_ylabel("Defect Rate")
    ax.set_ylim(0, 0.5)
    for bar, v in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                f"{v:.1%}", ha="center", fontweight="bold", fontsize=9)
    ax.legend(fontsize=9)

plt.suptitle("Categorical Feature Defect Rates", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap — single-column view focused on target
df_dum = pd.get_dummies(df, drop_first=True)
corr_target = df_dum.corr()[[TARGET]].sort_values(TARGET, ascending=False)

fig, ax = plt.subplots(figsize=(5, 7))
sns.heatmap(corr_target, annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-0.3, vmax=0.3, linewidths=0.5, ax=ax,
            annot_kws={"size": 10})
ax.set_title("Correlation with Surface Defect\n(after one-hot encoding)",
             fontweight="bold")
plt.tight_layout()
plt.show()


## Section 5 — Preprocessing

Gaussian Naive Bayes models each numeric feature as an independent Gaussian distribution per class — it does not require scaling, because it estimates the mean and variance of each feature directly from the data. No StandardScaler needed here, which is one of the model's practical advantages in production environments.

The two categorical variables (`container_type`, `part_protection`) must be **one-hot encoded**, since the model requires numeric inputs. We use `drop_first=True` to avoid dummy variable trap. All of this is bundled into a **sklearn Pipeline with ColumnTransformer** — a single object that handles preprocessing and prediction together, making the simulator in Section 10 trivially portable.

The train/test split uses `stratify=y` to preserve the ~22% defect rate in both sets — important when the target is imbalanced.


In [ ]:
X = df.drop(TARGET, axis=1)
y = df[TARGET]

# Stratified split — preserves ~22% defect rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

# Pipeline: ColumnTransformer (passthrough numerics, OHE categoricals) + GaussianNB
preprocessor = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), CAT_COLS)
])

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
print(f"Defect rate  : train={y_train.mean():.3f}  test={y_test.mean():.3f}")


## Section 6 — Train Model

We train a Gaussian Naive Bayes classifier as a probabilistic quality screening model. The architecture is a sklearn Pipeline — preprocessing and classification are fit together in one call, which ensures that any new data passed to `predict()` or `predict_proba()` goes through the exact same transformations without manual intervention.

Naive Bayes is simple on purpose. In quality operations, a model that returns an interpretable probability per part — and can be explained in terms of individual feature contributions — is more actionable than a complex ensemble that can't be audited.


In [ ]:
# Pipeline: transform + classify in one object
# This makes the simulator in Section 10 portable — no separate scaler to manage
model_nb = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", GaussianNB())
])

model_nb.fit(X_train, y_train)

print("Model trained successfully.")
print(f"Algorithm : GaussianNB")
print(f"Features  : {len(NUM_COLS)} numeric + {len(CAT_COLS)} categorical")
print(f"Classes   : {list(model_nb.classes_)} (0=Pass, 1=Defect)")


## Section 7 — Evaluate

With a 21.9% defect rate, a naive classifier that always predicts "pass" would score 78% accuracy — and catch zero defects. This is why we focus on the defect class specifically:

- **Recall (Defect)** — what fraction of actual defective parts does the model catch? A missed defect reaches the customer.
- **Precision (Defect)** — when the model flags a part as defective, how often is it right? Low precision means unnecessary rework.
- **AUC** — the model's ability to rank defective parts above non-defective ones, across all thresholds. This is the threshold-independent performance signal.

The confusion matrix and probability distribution plots reveal where the model's uncertainty concentrates.


In [ ]:
y_pred = model_nb.predict(X_test)
y_prob = model_nb.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print(f"Accuracy  : {acc:.4f}")
print(f"AUC-ROC   : {auc:.4f}")
print(f"Precision (Defect): {prec:.4f}")
print(f"Recall    (Defect): {rec:.4f}")
print(f"F1        (Defect): {f1:.4f}")
print("\n", classification_report(y_test, y_pred, target_names=["Pass", "Defect"]))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = ["Pass", "Defect"]

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted", fontweight="bold")
ax.set_ylabel("Actual", fontweight="bold")
ax.set_title("Confusion Matrix", fontweight="bold")
plt.tight_layout()
plt.show()
print("Rows = what actually happened. Columns = what the model predicted. Diagonal = correct predictions.")


In [ ]:
# ROC + Precision-Recall curves
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color="#4C9BE8", lw=2, label=f"AUC = {auc:.3f}")
axes[0].plot([0,1],[0,1], linestyle="--", color="gray", lw=1, label="Random baseline")
axes[0].fill_between(fpr, tpr, alpha=0.1, color="#4C9BE8")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve", fontweight="bold")
axes[0].legend()

axes[1].plot(rec_curve, prec_curve, color="#E8574C", lw=2, label=f"AP = {ap:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", lw=1,
                label=f"Baseline (defect rate = {y_test.mean():.2f})")
axes[1].fill_between(rec_curve, prec_curve, alpha=0.1, color="#E8574C")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Predicted probability distribution by true class
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6, color="#4C9BE8",
        label="Actual: Pass", edgecolor="white")
ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6, color="#E8574C",
        label="Actual: Defect", edgecolor="white")
ax.axvline(0.5, color="black", linestyle="--", lw=1.5, label="Decision threshold (0.5)")
ax.set_xlabel("Predicted Defect Probability")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution by True Class", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## Section 8 — Interpretability: Effect Size + Digital Pareto

Naive Bayes does not produce coefficients like Logistic Regression, but it has something more directly useful for quality engineers: **effect size**. Cohen's d measures how much the distribution of each numeric feature *shifts* between defect and non-defect parts. A large Cohen's d means that knowing the value of that variable meaningfully changes the defect probability. A near-zero d means the variable contributes almost nothing — even if it looked important on a boxplot.

For categorical variables, **Mutual Information** measures how much knowing the category value reduces uncertainty about the outcome — without assuming a linear relationship.

Together, these two measures build a **Digital Pareto** — the 80/20 visualization that answers *"which few conditions account for most of the defect risk?"* This is the language quality engineering teams speak.


In [ ]:
# Cohen's d — measures how much each numeric feature separates the two classes
def cohens_d(group0, group1):
    """Pooled-standard-deviation effect size. Larger absolute value = stronger class separation."""
    n0, n1 = len(group0), len(group1)
    s_pooled = np.sqrt(((n0-1)*group0.std()**2 + (n1-1)*group1.std()**2) / (n0+n1-2))
    return abs((group1.mean() - group0.mean()) / s_pooled)

cd_scores = {}
for col in NUM_COLS:
    g0 = df[df[TARGET] == 0][col]
    g1 = df[df[TARGET] == 1][col]
    cd_scores[col] = cohens_d(g0, g1)

cd_series = pd.Series(cd_scores).sort_values(ascending=False)

print("Cohen's d — Numeric Feature Effect Size (larger = stronger class separation):")
print()
for feat, d in cd_series.items():
    magnitude = ("Large" if d > 0.8 else
                 "Medium" if d > 0.5 else
                 "Small" if d > 0.2 else "Negligible")
    print(f"  {feat:<30} d = {d:.4f}   [{magnitude}]")


In [ ]:
# Mutual Information — categorical and numeric features together
# MI does not assume linearity — captures any statistical dependence
X_dum = pd.get_dummies(df.drop(TARGET, axis=1), drop_first=True)
mi = mutual_info_classif(X_dum, df[TARGET], random_state=RANDOM_STATE)
mi_series = pd.Series(mi, index=X_dum.columns).sort_values(ascending=False)

print("Mutual Information — All Features (including encoded categoricals):")
print(mi_series.round(4))


In [ ]:
# Digital Pareto — combines both metrics normalized to 0-100
# Shows which variables account for 80% of the measurable defect signal
cd_norm = (cd_series / cd_series.max() * 100)
mi_norm = (mi_series / mi_series.max() * 100) if mi_series.max() > 0 else mi_series

pareto = pd.concat([cd_norm, mi_norm]).sort_values(ascending=False)
pareto = pareto[~pareto.index.duplicated(keep='first')]  # dedup — keep highest score
pareto = pareto.sort_values(ascending=False)

cumulative = np.cumsum(pareto.values)
cumulative = cumulative / cumulative[-1] * 100

fig, ax1 = plt.subplots(figsize=(13, 6))

colors_pareto = ["#E8574C" if v >= 50 else "#4C9BE8" for v in pareto.values]
ax1.bar(range(len(pareto)), pareto.values, color=colors_pareto, edgecolor="white")
ax1.set_xticks(range(len(pareto)))
ax1.set_xticklabels(
    [f.replace("_", " ").title() for f in pareto.index],
    rotation=45, ha="right"
)
ax1.set_ylabel("Normalized Importance (%)", color="#333")
ax1.set_title("Digital Pareto — Factors Driving Surface Defect Risk",
              fontweight="bold", fontsize=13)

ax2 = ax1.twinx()
ax2.plot(range(len(pareto)), cumulative, color="#E8574C",
         marker="o", linewidth=2, label="Cumulative %")
ax2.axhline(80, color="green", linestyle="--", linewidth=1.2, label="80% threshold")
ax2.set_ylabel("Cumulative (%)", color="#E8574C")
ax2.set_ylim(0, 110)
ax2.legend(loc="center right")

plt.tight_layout()
plt.show()


### Summary of Practical Signals

The effect sizes tell a coherent quality story — and it points firmly at the **internal logistics stage**, not at injection or painting:

- **`pre_paint_storage_hrs` and `num_handlings`** are the top numeric drivers by Cohen's d. Parts that wait too long before entering the paint cabin, or that are touched and repositioned frequently, accumulate micro-scratches, dust, and surface contamination that paint cannot cover.
- **`part_protection`** (protected vs unprotected) is the dominant categorical driver by Mutual Information. Unprotected parts going through the same handling conditions show significantly higher defect rates — the protection layer is not cosmetic, it is functional.
- **`container_type`** also contributes — metal racks and cardboard pallets both outperform plastic boxes in defect rates, for different reasons: metal creates friction contact points; cardboard introduces dust and moisture.
- **Paint cabin variables** (`booth_humidity_pct`, `paint_viscosity`, `film_thickness_um`) show low Cohen's d individually — they amplify defects already present, rather than creating new ones.
- **Injection variables** (`regrind_pct`, `resin_temp_c`, `cooling_time_s`) are the weakest numeric drivers — within the simulated ranges, injection quality variation does not dominate the defect signal.

**Operational implication:** Quality interventions should focus first on handling protocols and protection coverage — before investing in injection or paint process optimization.


## Section 9 — Statistical Validation

Single train-test splits can be noisy — especially when the defect class has only ~22% representation. We use 5-fold stratified cross-validation and a 95% confidence interval to verify that the reported accuracy is stable, not a lucky partition.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_acc = cross_val_score(model_nb, X_train, y_train, cv=cv, scoring="accuracy")
cv_auc = cross_val_score(model_nb, X_train, y_train, cv=cv, scoring="roc_auc")
cv_f1  = cross_val_score(model_nb, X_train, y_train, cv=cv, scoring="f1")

print("5-Fold Stratified Cross-Validation (training set only):")
print(f"  Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"  AUC-ROC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"  F1       : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}")


In [ ]:
# 95% confidence interval for test-set accuracy
n = len(y_test)
p = accuracy_score(y_test, y_pred)
z = 1.96
margin = z * np.sqrt((p * (1 - p)) / n)

print(f"Test-set accuracy : {p:.4f}")
print(f"95% CI            : [{p - margin:.4f}, {p + margin:.4f}]")
print(f"Margin of error   : ±{margin:.4f}")
print()
print("Interpretation: If we repeatedly sampled 450 test cases from the same process,")
print("the model accuracy would fall within this range 95% of the time.")


## Section 10 — Part Inspection Simulator

A simple simulator turns the model into a pre-inspection decision tool. Before a batch of parts enters the paint cabin, an operator can input the current process conditions — storage time, handling count, container type, protection status — and receive an estimated defect probability. This is the most practical output of the model: not just a classification on historical data, but a real-time risk score that enables preventive action before the defect occurs.


In [ ]:
def simulate_part_inspection(
    regrind_pct: float = 10.0,
    resin_temp_c: float = 235.0,
    cooling_time_s: float = 18.0,
    paint_viscosity: float = 25.0,
    film_thickness_um: float = 30.0,
    booth_humidity_pct: float = 50.0,
    pre_paint_storage_hrs: float = 4.0,
    num_handlings: int = 3,
    container_type: str = "plastic_box",
    part_protection: str = "protected",
    threshold: float = 0.5
) -> tuple:
    """
    Estimate the surface defect probability for a single part.

    Parameters
    ----------
    regrind_pct            : % regrind material in resin (0–40)
    resin_temp_c           : Resin temperature °C (typical: 225–245)
    cooling_time_s         : Injection cooling time in seconds (typical: 12–24)
    paint_viscosity        : Paint viscosity at application (typical: 22–28)
    film_thickness_um      : Dry film thickness in microns (typical: 26–36)
    booth_humidity_pct     : Paint booth relative humidity % (typical: 35–65)
    pre_paint_storage_hrs  : Hours part waited before entering paint cabin (0–24)
    num_handlings          : Number of manual handlings during transport (1–8)
    container_type         : 'plastic_box', 'metal_rack', or 'cardboard_pallet'
    part_protection        : 'protected' or 'unprotected'
    threshold              : Classification cutoff (default 0.5)
    """
    df_new = pd.DataFrame([{
        "regrind_pct": regrind_pct,
        "resin_temp_c": resin_temp_c,
        "cooling_time_s": cooling_time_s,
        "paint_viscosity": paint_viscosity,
        "film_thickness_um": film_thickness_um,
        "booth_humidity_pct": booth_humidity_pct,
        "pre_paint_storage_hrs": pre_paint_storage_hrs,
        "num_handlings": num_handlings,
        "container_type": container_type,
        "part_protection": part_protection
    }])

    # Pipeline handles encoding internally — same preprocessing as training
    prob  = model_nb.predict_proba(df_new)[0, 1]
    clase = int(prob >= threshold)

    print("=" * 54)
    print("  SURFACE DEFECT PROBABILITY SIMULATOR")
    print("=" * 54)
    print(f"  Container type       : {container_type.upper()}")
    print(f"  Part protection      : {part_protection.upper()}")
    print(f"  Pre-paint storage    : {pre_paint_storage_hrs:.1f} hrs")
    print(f"  Number of handlings  : {num_handlings}")
    print(f"  Regrind %            : {regrind_pct:.1f}%")
    print(f"  Resin temp           : {resin_temp_c:.1f} °C")
    print(f"  Paint viscosity      : {paint_viscosity:.1f}")
    print(f"  Film thickness       : {film_thickness_um:.1f} µm")
    print(f"  Booth humidity       : {booth_humidity_pct:.1f}%")
    print("-" * 54)
    print(f"  Defect probability   : {prob:.3f} ({prob*100:.1f}%)")
    print(f"  Decision threshold   : {threshold:.2f}")
    if clase == 1:
        print("  Prediction: >>> REJECT — HIGH DEFECT RISK <<<")
    else:
        print("  Prediction: >>> PASS — LOW DEFECT RISK <<<")
    print("=" * 54)
    return prob, clase


### Scenario A — Low-risk: controlled conditions throughout the process


In [ ]:
simulate_part_inspection(
    regrind_pct=5.0,
    resin_temp_c=235.0,
    cooling_time_s=20.0,
    paint_viscosity=25.0,
    film_thickness_um=30.0,
    booth_humidity_pct=50.0,
    pre_paint_storage_hrs=1.0,   # short wait — low contamination risk
    num_handlings=2,             # minimal handling
    container_type="plastic_box",
    part_protection="protected", # protected throughout
    threshold=0.5
)


### Scenario B — High-risk: poor handling, long storage, no protection


In [ ]:
simulate_part_inspection(
    regrind_pct=28.0,
    resin_temp_c=248.0,          # out of spec temperature
    cooling_time_s=13.0,         # insufficient cooling
    paint_viscosity=21.0,        # low viscosity — uneven coverage
    film_thickness_um=38.0,      # over-spec film
    booth_humidity_pct=68.0,     # high humidity — adhesion issues
    pre_paint_storage_hrs=18.0,  # very long pre-paint wait
    num_handlings=6,             # excessive handling
    container_type="metal_rack", # metal contact — scratch risk
    part_protection="unprotected",
    threshold=0.5
)


### Scenario C — Intervention: same high-risk profile but adding part protection


In [ ]:
# Same as Scenario B — only changing part_protection to 'protected'
# This isolates the marginal effect of adding protection on a high-risk profile
simulate_part_inspection(
    regrind_pct=28.0,
    resin_temp_c=248.0,
    cooling_time_s=13.0,
    paint_viscosity=21.0,
    film_thickness_um=38.0,
    booth_humidity_pct=68.0,
    pre_paint_storage_hrs=18.0,
    num_handlings=6,
    container_type="metal_rack",
    part_protection="protected",  # only change
    threshold=0.5
)


## Final Reflection

This project is not about predicting surface defects with certainty.  
It is about **making defect risk visible before the damage is done**.

Surface defects in painted plastic parts are deceptive: they appear at final inspection, but they originate at injection, accumulate during transport, and get revealed by the paint. By the time a defect is detected visually, the part has already traveled through three or four process stages. Traditional inspection catches defects after the fact — this model shifts the conversation upstream.

What this model offers is not a verdict, but a **conversation**:
- *What happens if we add protection to all parts going to metal racks?*
- *What is the breakeven between reducing pre-paint waiting time vs investing in booth humidity control?*
- *Where is the handling count threshold where defect probability jumps meaningfully?*

These questions matter more than the AUC score.

---

*LozanoLsa  |  Operational Excellence · MBB · Machine Learning  |  GitHub: LozanoLsa*
